# Ensemble Learning

Many weak models together create a stronger model.

Q. If all trees see exactly the same data, won't they all be identical?

- Then averaging gives nothing.

```
T1 = 50
T2 = 50
T3 = 50
T4 = 50
T5 = 50
```

Average = 50

Still one tree.

## Bootstrap Sampling

Data = [A, B, C, D, E]

We use Randomly sample with replacement.

```
Tree 1: [A, C, E, A, D]
Tree 2: [B, B, D, E, C]
Tree 3: [C, A, A, E, E]
```

**Notice:**
- some rows repeated
- some rows missing

Every tree gets a slightly different dataset.

## Feature Randomness

```
fet = [
  Area,
  Bedrooms,
  Age,
  Distance
]
```

A Decision Tree may always choose: **Area** at the root.

- we choose random subset of features

```
Tree 1 root can only choose from: [Area, Age]
Tree 2 root can only choose from: [Bedrooms, Distance]
Tree 3 root can only choose from: [Area, Bedrooms]
```

- Now trees become much more diverse.

## Let's build Random Forest Regressor:

In [2]:
import random
from decision_tree import DecisionTreeRegressor


class RandomForestRegressor:

    def __init__(
        self,
        n_estimators=10,
        max_depth=5,
        min_samples_split=2,
        max_features=None,
        random_state=None
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def bootstrap_sample(self, X, y):
        n_samples = len(X)

        X_sample = []
        y_sample = []

        for _ in range(n_samples):
            idx = random.randint(0, n_samples - 1)

            X_sample.append(X[idx])
            y_sample.append(y[idx])

        return X_sample, y_sample

    def fit(self, X, y):

        if self.random_state is not None:
            random.seed(self.random_state)
        
        self.trees = []

        for _ in range(self.n_estimators):
            X_sample, y_sample = self.bootstrap_sample(X, y)

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )

            tree.fit(
                X_sample,
                y_sample
            )

            self.trees.append(tree)

    def predict(self, X):
        all_predictions = []

        for tree in self.trees:
            all_predictions.append(
                tree.predict(X)
            )

        final_predictions = []

        for sample_idx in range(len(X)):
            total = 0

            for tree_idx in range(len(self.trees)):
                total += all_predictions[tree_idx][sample_idx]

            final_predictions.append(
                total / len(self.trees)
            )

        return final_predictions

In [3]:
def root_mean_squared_error(y_true, y_pred):
    n = len(y_true)
    mse = sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred)) / n
    return round(mse ** 0.5, 2)

In [10]:
X = [
    [500, 1],
    [550, 1],
    [600, 1],
    [650, 2],
    [700, 2],
    [800, 2],
    [900, 2],
    [1200, 3],
    [1300, 3],
    [1400, 3],
    [1500, 3],
    [1600, 3],
    [1700, 4],
    [1800, 4],
    [1900, 4],
    [2000, 4]
]

y = [48, 54, 57, 64, 66, 95, 81, 120, 127, 136, 149, 157, 171, 205, 189, 197]

rf = RandomForestRegressor(
    n_estimators=5,
    max_depth=3,
    min_samples_split=2,
    max_features=1,
    random_state=42
)

rf.fit(X, y)

predictions = rf.predict([
    [500, 1],
    [1600, 3]
])

print(predictions)

[53.3, 151.89166666666668]


In [11]:
print(
    rf.predict([
        [575, 1],
        [1650, 3]
    ])
)

[54.9, 151.89166666666668]


In [12]:
train_error = root_mean_squared_error(y, rf.predict(X))
print("Train RMSE:", train_error)

test_error = root_mean_squared_error([55,70,132,165,192], rf.predict([[575, 1],[750, 2],[1350, 3],[1650, 3],[1950, 4]]))
print("Test RMSE:", test_error)

Train RMSE: 8.25
Test RMSE: 11.3


In [19]:
X = [
    [500, 1, 3],
    [600, 1, 1],
    [800, 2, 2],
    [900, 3, 2],
    [1100, 2, 3],
    [1300, 3, 1],
    [1500, 3, 1],
    [1600, 3, 3],
    [1700, 4, 2],
    [1800, 4, 3]
]

y = [50,95,85,120,115,135,150,175,180,170]

rf1 = RandomForestRegressor(
    n_estimators=10,
    max_depth=2,
    min_samples_split=2,
    max_features=3,
    random_state=42
)

rf1.fit(X, y)

print(
    rf1.predict([
        [500, 1, 3],
        [1600, 3, 3]
    ])
)

[69.66666666666667, 159.23214285714283]


In [20]:
print(
    rf1.predict([
        [550, 1, 1],
        [1650, 3, 2]
    ])
)

[97.92380952380952, 159.73214285714283]


In [21]:
train_error = root_mean_squared_error(y, rf1.predict(X))
print("Train RMSE:", train_error)
test_error = root_mean_squared_error([75,100,125,170,175], rf1.predict([[550, 1, 2],[850, 2, 2],[1200, 3, 2],[1650, 3, 2],[1750, 4, 3]]))
print("Test RMSE:", test_error)

Train RMSE: 9.91
Test RMSE: 12.84
